In [1]:
import os
import json

import sys
sys.path.append(os.path.dirname(os.path.abspath("")))
from utils.utils import set_seed
# set_seed(42)

In [12]:
config_file_name = "config_crc.json"  # config_crc.json, config_varseek_crc.json, config_demo.json
gpu_id = 4

Dataset:

- Manuscript: https://www.nature.com/articles/s41588-025-02193-3#data-availability
- 10x: https://www.10xgenomics.com/platforms/visium/product-family/dataset-human-crc
- 10x (alt): https://www.10xgenomics.com/datasets/visium-hd-cytassist-gene-expression-libraries-of-human-crc
- GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE280318
- SRA: https://www.ncbi.nlm.nih.gov/Traces/study/?acc=PRJNA1177833&o=acc_s%3Aa

## Download demo checkpoint

A previously saved checkpoint is provided for this demo. This downloads to `experiments/demo`:

In [6]:
os.chdir("../")

In [ ]:
with open(f"configs/{config_file_name}", "r") as f:
    config_data = json.load(f)

load_dir = config_data["experiment_dirs"]["load_dir"]
test_log_path = f"experiments/{load_dir}/test_out.txt"

In [14]:
box_links = {
    "config_crc.json": "https://caltech.box.com/shared/static/xcy1ouap4d3hipqibh2pd9fkzmdlt2d6.gz",
    "config_varseek_crc.json": "xxxxx",  #!!! replace
}

if config_file_name == "config_demo.json":
    !gdown --folder https://drive.google.com/drive/folders/1ecTOXmSeQU9v8aKYniQQab2QqkOlIl8u?usp=drive_link
else:
    load_path = f"experiments/{load_dir}"
    if not os.path.exists(load_path) or not os.listdir(load_path):
        !wget -O {load_path}.tar.gz {box_links[config_file_name]}
        !tar -xzvf {load_path}.tar.gz

## Config file

Parameters are defined in a config file (``./configs/config_varseek_crc.json`` for this demo). Important parameters include:

- ``comps``: these need to be consistent with the settings used during training. For the demo checkpoint, all components were used.
- ``cell_types``: also need to be consistent with the settings used during training.
- ``data_sources_predict``: locations of data for prediction. For your own data ensure to update`fp_hist` and `fp_nuc_seg`.
- ``regions_predict.divisions``: By default, the whole image will be used for prediction.
- ``experiment_dirs.load_dir``: The experiment ID to load the checkpoint from (``demo`` for this demo, or set to ``latest`` to use the latest experiment by timestamp)

## Get predictions

```sh
python inference.py --config_file configs/FILENAME.json --epoch EPOCH --mode predict --fold_id FOLD --gpu_id GPU_NUM
```

- ``--config_file`` path to config file
- ``--epoch`` specifies which epoch to use, e.g., ``10`` to use the model from epoch 10, or use `last` for the most recent, or `all` for all epochs
- ``--fold_id`` specifies the cross-validation fold (1, 2, 3...) the model was trained from
- ``--gpu_id`` which GPU to use (0, 1, 2...)

In [ ]:
!python inference.py --config_file configs/{config_file_name} --epoch last --mode predict --fold_id 1 --gpu_id {gpu_id} | tee {test_log_path}

Using GPUs: 4
['CD19+CD20+ B', 'CD4+ T cells', 'CD8+ T cells', 'CMS1', 'CMS2', 'CMS3', 'CMS4', 'Enteric glial cells', 'Goblet cells', 'IgA+ Plasma', 'IgG+ Plasma', 'Intermediate', 'Lymphatic ECs', 'Mast cells', 'Mature Enterocytes type 1', 'Mature Enterocytes type 2', 'Myofibroblasts', 'NK cells', 'Pericytes', 'Pro-inflammatory', 'Proliferating', 'Proliferative ECs', 'Regulatory T cells', 'SPP1+', 'Smooth muscle cells', 'Stalk-like ECs', 'Stem-like/TA', 'Stromal 1', 'Stromal 2', 'Stromal 3', 'T follicular helper cells', 'T helper 17 cells', 'Tip-like ECs', 'Unknown', 'cDC', 'gamma delta T cells']
Num cell types 36
422 genes
Avgexp shape  (21, 422)
Histology image (30874, 31470, 3), Nuclei (30874, 31470)
215553 cells
Patches min/max coords 0 30874
Getting valid patches
100%|████████████████████████████████████| 19180/19180 [01:21<00:00, 236.06it/s]
Standardisation
Predict using experiments/crc_without_varseek/models/epoch_50_model.pth
100%|███████████████████████████████████████| 2002/2

## Outputs

The predictions were saved to ``experiments/{run_name}/predict_output/``, and the csv files contain the predicted gene expressions for each cell, where the index is the cell ID that corresponds to the IDs from the nuclei segmentation image, and the columns are the genes. An example is provided as ``example_output.csv`` to show the format.  